# Data Collection Using SpaceX REST API

This notebook retrieves historical launch records from the public SpaceX API and saves a reproducible JSON/CSV snapshot. API fields describe the mission/launch; landing labels for the course classification are sourced from the IBM teaching dataset in later notebooks.

In [ ]:
import json, requests, pandas as pd
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA = ROOT / 'data'; DATA.mkdir(exist_ok=True)
url = 'https://api.spacexdata.com/v4/launches/past'
response = requests.get(url, timeout=45)
response.raise_for_status()
launches = response.json()
print('Launch records retrieved:', len(launches))

In [ ]:
records = []
for item in launches:
    records.append({
        'flight_number': item.get('flight_number'),
        'name': item.get('name'),
        'date_utc': item.get('date_utc'),
        'success': item.get('success'),
        'rocket_id': item.get('rocket'),
        'launchpad_id': item.get('launchpad'),
        'payload_ids': ';'.join(item.get('payloads') or []),
        'core_ids': ';'.join((c.get('core') or '') for c in (item.get('cores') or []) if c.get('core')),
        'landing_attempts': sum(c.get('landing_attempt') is True for c in (item.get('cores') or [])),
        'landing_successes': sum(c.get('landing_success') is True for c in (item.get('cores') or [])),
    })
api_df = pd.DataFrame(records)
api_df.to_csv(DATA / 'spacex_api_launches.csv', index=False)
(DATA / 'spacex_api_launches.json').write_text(json.dumps(launches, indent=2), encoding='utf-8')
api_df.head()

## Data quality checks

Review duplicate flight numbers, date parsing, and missing success/landing fields before using API data in a separate analysis.

In [ ]:
api_df['date_utc'] = pd.to_datetime(api_df['date_utc'], errors='coerce', utc=True)
print('Duplicate flight numbers:', api_df['flight_number'].duplicated().sum())
print('Missing mission success:', api_df['success'].isna().sum())
print('Missing dates:', api_df['date_utc'].isna().sum())
api_df.describe(include='all').transpose().head(12)